# Curse of Dimensionality · 深入实验（08–11）

目标：**区分高环境维度、低内在维度、谱有效秩和语义检索质量。** 每组实验都要预测、运行、分析反例。

在仓库根目录 `pip install -e '[notebook]'`，再启动 Jupyter。纯 NumPy / sklearn 即可运行前三组；Autoencoder 需额外安装 `[deep]`。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from curse_of_dim.advanced import (swiss_roll, effective_rank, knn_overlap,
    synthetic_embeddings, embedding_transformations, retrieval_metrics)
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap
np.set_printoptions(precision=3, suppress=True)

## 08 | 流形：ambient 维度 vs intrinsic 维度

预测：原始64维空间虽维数大，但由于是3D等距嵌入，近邻基本不变；PCA压到2维可能把相隔一圈的点折叠在一起。Isomap的图最短路可能帮助展开，但它也依赖 `n_neighbors`。

**仅替换一个变量：** 把 `neighbors=8` 改成 6/12/24，观察 Isomap 对短路边的敏感性。

In [ ]:
x, oracle, t = swiss_roll(400, ambient=64, seed=808)
nt = 300
pca = PCA(n_components=2).fit(x[:nt])
iso = Isomap(n_components=2, n_neighbors=10).fit(x[:nt])
p = pca.transform(x)
i = np.vstack((iso.embedding_, iso.transform(x[nt:])))
for name, emb in [('ambient',x),('PCA2',p),('Isomap2',i)]:
    print(name, 'overlap@10 =', round(knn_overlap(oracle[:nt],oracle[nt:], emb[:nt],emb[nt:],10),3))
fig, ax = plt.subplots(1,2,figsize=(11,4))
ax[0].scatter(p[:,0],p[:,1],c=t,s=8,cmap='viridis'); ax[0].set_title('PCA');
ax[1].scatter(i[:,0],i[:,1],c=t,s=8,cmap='viridis'); ax[1].set_title('Isomap');
plt.show()

## 09 | Embedding 主成分：相同处理，两个相反结论

两种随机世界都**有确定标签**，但只有一种的主成分是无关干扰。

观察 raw / centered / remove_top_pc / whiten_top_components 的 top-1 accuracy、Precision@5 与 hubness Gini。预测：nuisance 场景移除 PC1 有帮助；signal 场景反之。

In [ ]:
for scenario in ('nuisance','signal'):
    docs, queries, yl, yq = synthetic_embeddings(scenario=scenario)
    print('\n',scenario, 'spectral rank=',round(effective_rank(docs)['entropy_rank'],2))
    for name,(d,q) in embedding_transformations(docs,queries).items():
        m=retrieval_metrics(d,q,yl,yq,k=5)
        print(f"{name:23s} top1={m['top1_accuracy']:.3f} p@5={m['precision_at_k']:.3f} hub_gini={m['hubness_gini']:.3f}")

## 10 | 有效秩不是真实流形维度

令 `Z∈R⁶`，`X = ZQᵀ + σ ε ∈R¹²⁸`，改变 `σ` 而不改变潜在自由度。比较有效秩和近邻恢复。这个实验能解释为什么仅凭协方差谱判断“真实维度”会受到噪声影响。

In [ ]:
rng=np.random.default_rng(1001)
n, nq, dim, rank=320,80,128,6
z=rng.normal(size=(n+nq,rank))
q,_=np.linalg.qr(rng.normal(size=(dim,rank)))
noise=rng.normal(size=(n+nq,dim))
for s in [0,.05,.1,.2,.4,.6]:
    x=z@q.T+s*noise
    p=PCA(n_components=rank).fit(x[:n]).transform(x)
    a=knn_overlap(z[:n],z[n:],x[:n],x[n:],10)
    b=knn_overlap(z[:n],z[n:],p[:n],p[n:],10)
    print(f'noise={s:.2f}: erank={effective_rank(x[:n])["entropy_rank"]:.1f}, raw_overlap={a:.3f}, PCA_overlap={b:.3f}')

## 11 | 可选 Torch AE：重构不代表展开

运行下面的代码需要 `pip install -e '[deep]'`。预先固定随机种子；代码的训练、评估逻辑位于 `advanced.experiment_11_autoencoder`。比较**留出重构MSE**与**真实流形近邻 overlap@10**，它们优化目标不同。

In [ ]:
# Optional (CPU, ~seconds to minutes depending on hardware):
# from curse_of_dim.advanced import experiment_11_autoencoder
# report = experiment_11_autoencoder('outputs/autoencoder')
# print(report)

## 本地真实向量 audit

真实数据分离 query/doc；可选标签决定能否估计检索相关性。下方生成**纯合成样例**；不要将它当作真实模型效果。

In [ ]:
from pathlib import Path
from curse_of_dim.audit import audit_npz
d,q,dl,ql=synthetic_embeddings(n_docs=240,n_queries=60,ambient=64)
Path('outputs').mkdir(exist_ok=True)
np.savez_compressed('outputs/demo_embeddings.npz',doc_embeddings=d, query_embeddings=q, doc_labels=dl, query_labels=ql)
audit=audit_npz('outputs/demo_embeddings.npz','outputs/notebook_audit', k=5)
print('raw:',audit['transforms']['raw']);print('remove_top_pc:',audit['transforms']['remove_top_pc'])

### 最终思考题

1. 为什么 64D 的点可以比压到 2D 的点更准确地保留局部邻居？
2. 什么条件下 PCA-2 与 Isomap-2 会各自失败？Isomap 有没有“短路边”的问题？
3. PCA 的谱有效秩为何不等于流形维度？怎么区分环境噪声与新的潜在因子？
4. 为什么基于文档主成分的 `remove_top_pc` 必须在独立 query 上验证？
5. 当一个方法显著改善了 hubness，却降低 Precision@5，应当如何评估？

论文 / 数学公式 / 实验控制变量详见 `docs/EMBEDDING_GEOMETRY.md`。